# RAG-Augmented Distillation — Kaggle GPU Run (2×T4, 30GB)

Runs the complete RAD pipeline end-to-end on Kaggle's 2×T4 (30GB total VRAM).

**Pipeline:**
1. Build ChromaDB from SQuAD v2 contexts (~15 min, CPU)
2. Generate teacher soft labels — RAG + bare + negative logits (~30 min, GPU)
3. Train RAD student — flan-t5-small with L_RAD loss (~45 min, GPU)
4. Evaluate — bare teacher vs RAG teacher vs RAD student

**Key result to look for:** Student (RAD) EM/F1 should exceed a standard-KD student,
demonstrating that RAG-augmented soft labels transfer retrieved knowledge into student weights.

> **Before running:** Settings → Accelerator → **GPU T4 x2** | Settings → Internet → **On**

In [ ]:
import subprocess, os, sys
import torch

# Verify GPU
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Enable: Settings → Accelerator → GPU T4 x2")

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB")
print(f"Total VRAM: {sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus))/1e9:.1f} GB")

In [ ]:
# Clone repo
REPO_DIR = '/kaggle/working/model-distillation'
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/aabhimittal/model-distillation-', REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)  # add repo root to Python path (avoids pip -e and its resolver)
print(f"Repo: {os.getcwd()}")

# Install ONLY the packages Kaggle doesn't already ship.
# Kaggle already has: torch, transformers, datasets, numpy, tqdm, pyyaml, accelerate, nltk
# We only need chromadb, sentence-transformers, evaluate.
# --no-deps prevents the resolver from touching pre-installed Kaggle packages,
# which avoids the dask-cuda / cuml / google-adk conflict warnings.
pkgs = ['chromadb', 'sentence-transformers', 'evaluate']
subprocess.run(['pip', 'install', '--quiet', '--no-deps'] + pkgs, check=True)
# chromadb 1.5.x transitive deps not pulled in by --no-deps:
subprocess.run(
    ['pip', 'install', '--quiet',
     'hnswlib', 'chroma-hnswlib', 'mmh3', 'overrides',
     'posthog', 'pypika', 'tenacity', 'tokenizers',
     # added in chromadb 1.5.9:
     'bcrypt>=4.0.1', 'kubernetes>=28.1.0', 'onnxruntime>=1.14.1',
     'opentelemetry-exporter-otlp-proto-grpc>=1.2.0', 'pybase64>=1.4.1'],
    check=True
)
print("Packages ready.")

In [ ]:
# Paths — persisted under /kaggle/working for the full session
BASE_DIR        = '/kaggle/working/rad'
CHROMA_DIR      = f'{BASE_DIR}/chroma_db'
SOFT_LABELS_DIR = f'{BASE_DIR}/soft_labels'
OUTPUT_DIR      = f'{BASE_DIR}/outputs'

for d in [CHROMA_DIR, SOFT_LABELS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Use device_map=auto to distribute teacher across both T4s during label generation
DEVICE_MAP = 'auto' if torch.cuda.device_count() > 1 else None
print(f"device_map : {DEVICE_MAP}")
print(f"ChromaDB   : {CHROMA_DIR}")
print(f"Soft labels: {SOFT_LABELS_DIR}")
print(f"Outputs    : {OUTPUT_DIR}")

## Phase 1 — Build ChromaDB vector store

Embeds ~50k unique SQuAD v2 Wikipedia passages into ChromaDB using
`sentence-transformers/all-MiniLM-L6-v2` (CPU, ~15 min).
Idempotent — safe to re-run, skips if already populated.

In [ ]:
subprocess.run([
    'python', 'scripts/build_vector_db.py',
    '--config', 'configs/distillation_config.yaml',
    '--chroma-dir', CHROMA_DIR,
], check=True)

## Phase 2 — Generate teacher soft labels

Runs `flan-t5-base` on all 10k training questions (3 passes per question):
- **RAG pass** — teacher sees top-3 retrieved passages → logits for `L_RAG`
- **Bare pass** — teacher sees no context → logits for `L_KL`
- **Negative pass** — teacher sees irrelevant context → logits for `L_CRA`

Saves `.npz` files to `SOFT_LABELS_DIR`. Idempotent — skips existing files.
`--device-map auto` distributes teacher layers across both T4 GPUs.

In [ ]:
cmd = [
    'python', 'scripts/generate_soft_labels.py',
    '--config', 'configs/distillation_config.yaml',
    '--soft-labels-dir', SOFT_LABELS_DIR,
    '--chroma-dir', CHROMA_DIR,
]
if DEVICE_MAP:
    cmd += ['--device-map', DEVICE_MAP]
subprocess.run(cmd, check=True)

In [ ]:
# Sanity check — verify retrieval actually shifts the teacher's distribution
import glob, numpy as np
npz_files = sorted(glob.glob(f'{SOFT_LABELS_DIR}/*.npz'))
print(f"Soft label files: {len(npz_files)}")

sample = np.load(npz_files[0])
print(f"Keys: {list(sample.keys())}")
for k in sample.keys():
    print(f"  {k}: shape={sample[k].shape}, dtype={sample[k].dtype}")

# KL(RAG || bare) > 0 means retrieval meaningfully shifted the teacher distribution
p_rag  = np.exp(sample['rag_logits']  - sample['rag_logits'].max(-1, keepdims=True))
p_bare = np.exp(sample['bare_logits'] - sample['bare_logits'].max(-1, keepdims=True))
p_rag  /= p_rag.sum(-1, keepdims=True)
p_bare /= p_bare.sum(-1, keepdims=True)
kl = np.mean(np.sum(p_rag * np.log(p_rag / (p_bare + 1e-9) + 1e-9), axis=-1))
print(f"\nMean KL(RAG || bare): {kl:.4f}")
print("↑ Should be > 0 — confirms retrieval shifts the teacher's output distribution.")

## Phase 3 — Train the RAD student

Trains `flan-t5-small` with the full L_RAD loss for 3 epochs (10k examples, effective batch=32).

```
L_RAD = 0.5·L_RAG + 0.2·L_KL + 0.1·L_CRA + 0.2·L_CE
```

Checkpoints saved every 500 steps. All 4 loss components logged every 50 steps.

In [ ]:
# Student trains on a single GPU (gradient flow requires unified device)
subprocess.run([
    'python', 'scripts/train_student.py',
    '--config', 'configs/distillation_config.yaml',
    '--output-dir', OUTPUT_DIR,
    '--soft-labels-dir', SOFT_LABELS_DIR,
    '--chroma-dir', CHROMA_DIR,
], check=True)

In [ ]:
# Plot loss curves
import json, matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/loss_history.json') as f:
    history = json.load(f)

steps = [h['step'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(steps, [h['loss'] for h in history], color='black', linewidth=1.5)
axes[0].set_title('Total RAD Loss')
axes[0].set_xlabel('Global Step')
axes[0].grid(alpha=0.3)

for key, label, color in [
    ('L_RAG', 'L_RAG — vs RAG-teacher (novel)', 'tab:blue'),
    ('L_KL',  'L_KL  — vs bare teacher (std KD)', 'tab:orange'),
    ('L_CRA', 'L_CRA — contrastive retrieval', 'tab:green'),
    ('L_CE',  'L_CE  — hard label CE', 'tab:red'),
]:
    axes[1].plot(steps, [h[key] for h in history], label=label, color=color, linewidth=1.2)
axes[1].set_title('Loss Components')
axes[1].set_xlabel('Global Step')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 4 — Evaluate all conditions

Compares three conditions on 1k SQuAD v2 validation examples:
| Condition | Model | Retrieval at inference? |
|---|---|---|
| Teacher (bare) | flan-t5-base | No |
| Teacher + RAG | flan-t5-base | Yes (ChromaDB) |
| **Student (RAD)** | flan-t5-small | **No** ← the point |

Expected: Student (RAD) EM/F1 ≈ Teacher (bare), and significantly > a standard-KD student.

In [ ]:
subprocess.run([
    'python', 'scripts/evaluate.py',
    '--config', 'configs/distillation_config.yaml',
    '--student-rad', f'{OUTPUT_DIR}/final',
    '--output-dir', OUTPUT_DIR,
    '--chroma-dir', CHROMA_DIR,
], check=True)

In [ ]:
# Results table + bar chart
import json, numpy as np, matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/eval_results.json') as f:
    results = json.load(f)

print(f"\n{'='*62}")
print(f"  RAG-AUGMENTED DISTILLATION — RESULTS (SQuAD v2)")
print(f"{'='*62}")
print(f"{'Model':<30} {'EM':>8} {'F1':>8} {'BLEU-4':>8}")
print(f"{'-'*62}")
for name, m in results.items():
    print(f"{name:<30} {m.get('exact_match',0)*100:>7.1f}%  {m.get('f1',0)*100:>7.1f}%  {m.get('bleu4',0)*100:>7.1f}%")
print(f"{'='*62}")

models    = list(results.keys())
em_scores = [results[m].get('exact_match', 0) * 100 for m in models]
f1_scores = [results[m].get('f1', 0) * 100 for m in models]

x, w = np.arange(len(models)), 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, em_scores, w, label='Exact Match (%)', color=['#4C72B0','#DD8452','#55A868'])
b2 = ax.bar(x + w/2, f1_scores,  w, label='F1 (%)',          color=['#6F9DC8','#E8A87C','#77C28A'])
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=10, ha='right', fontsize=11)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('RAG-Augmented Distillation — Model Comparison (SQuAD v2)', fontsize=13)
ax.legend(fontsize=11)
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylim(0, max(f1_scores) * 1.25)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAll outputs saved to: {OUTPUT_DIR}")